### Beyond the Squeaky Wheel: 311 Engagement & Equity Analysis
### Notebook 5: Service Need Index (SNI) Data Preparation

Assembles and calculates raw per-tract values for each SNI physical and land-use metric (roads, transit, impervious surface, zoning, building age, job density, and vacancy). Merges all metrics onto the base census tract geography and exports a master GeoPackage, CSV, and summary table.

In [ ]:
# Step 1: Import libraries

import os
import gc
import glob
from datetime import datetime
import pandas as pd
import geopandas as gpd
import numpy as np
from tqdm import tqdm

In [ ]:
# Step 2: Set up file paths and upload data

BASE_TRACTS_PATH = "INSERT FILE PATH: base census tract geometries GeoPackage"

MAJOR_ROADS_PATH = "INSERT FILE PATH: major roads GeoPackage"
MINOR_ROADS_PATH = "INSERT FILE PATH: minor roads GeoPackage"
RAILWAYS_PATH = "INSERT FILE PATH: railways GeoPackage"
IMPERVIOUS_CSV_PATH = "INSERT FILE PATH: impervious surface CSV"
LODES_CSV_PATH = "INSERT FILE PATH: LODES job density CSV"
HOUSING_AGE_CSV_PATH = "INSERT FILE PATH: median housing age CSV"
OVERTURE_POI_PATH = "INSERT FILE PATH: Overture POI GeoPackage"
TRANSIT_LINES_PATH = "INSERT FILE PATH: transit rail lines GeoPackage"
ZONING_DIR = "INSERT FOLDER PATH: per-city zoning GeoPackages" 

OUTPUT_DIR = "INSERT FOLDER PATH: SNI data prep outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MASTER_GPKG_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "SNI_master_tract_variables.gpkg")
MASTER_CSV_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "SNI_master_tract_variables.csv")
PREP_SUMMARY_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "SNI_data_prep_summary.csv")

TARGET_CRS = "EPSG:5070"  # project standard (NAD83 / Conus Albers)

In [ ]:
# Sanity check - confirm file uploads

input_files = {
    "Base tracts": BASE_TRACTS_PATH,
    "Major roads": MAJOR_ROADS_PATH,
    "Minor roads": MINOR_ROADS_PATH,
    "Railways": RAILWAYS_PATH,
    "Transit Lines": TRANSIT_LINES_PATH,    
    "Impervious surface CSV": IMPERVIOUS_CSV_PATH,
    "LODES CSV": LODES_CSV_PATH,
    "Housing age CSV": HOUSING_AGE_CSV_PATH,
    "Overture POI": OVERTURE_POI_PATH,
    #"Zoning (classified)": ZONING_CLASSIFIED_PATH,
}

for label, path in input_files.items():
    exists = os.path.exists(path)
    tqdm.write(f"{'FOUND    ' if exists else 'MISSING  '} {label}: {path}")


zoning_files = glob.glob(os.path.join(ZONING_DIR, "*.gpkg"))
tqdm.write(f"{'FOUND    ' if zoning_files else 'MISSING  '} Zoning: {len(zoning_files)} city file(s) in {ZONING_DIR}")

In [ ]:
# Step 3A: Configure data parameters

# ---- Configuration ----
GEOID_DTYPE = str  # always read/join GEOID as string, zfill(11), per project convention

ZONING_CATEGORY_COLUMN = "standard_category"
SNI_ZONING_CATEGORIES = ["industrial", "commercial", "mixed_use"]

# Column names expected in the source CSVs -- verify these against your actual files
IMPERVIOUS_VALUE_COL = "Impervious_Prcnt"
LODES_VALUE_COL = "total_jobs"
HOUSING_YEAR_COL = "median_year_built"

CURRENT_YEAR = datetime.now().year

In [ ]:
# Step 3B: count zoning categories (including blank/null) before processing

zoning_check_parts = []
for path in glob.glob(os.path.join(ZONING_DIR, "*.gpkg")):
    part = gpd.read_file(path, engine="pyogrio", columns=[ZONING_CATEGORY_COLUMN], read_geometry=False)
    zoning_check_parts.append(part)

zoning_check = pd.concat(zoning_check_parts, ignore_index=True)

n_null = zoning_check[ZONING_CATEGORY_COLUMN].isna().sum()
n_blank_string = (zoning_check[ZONING_CATEGORY_COLUMN].astype(str).str.strip() == "").sum()

print(f"Total zoning records: {len(zoning_check):,}")
print(f"Null (NaN) values: {n_null:,}")
print(f"Blank/whitespace string values: {n_blank_string:,}")
print()
print("Full category breakdown (including blanks/nulls):")
print(zoning_check[ZONING_CATEGORY_COLUMN].value_counts(dropna=False))

In [ ]:
# Step 4: count and clean filler data in housing years
        # needed before any raw US Census data that is used in any code

ACS_SENTINEL_VALUES = [-666666666, -999999999, -888888888, -555555555, -333333333, -222222222]

housing_age_df = pd.read_csv(HOUSING_AGE_CSV_PATH, dtype={"GEOID": str})
housing_age_df["GEOID"] = housing_age_df["GEOID"].str.zfill(11)
housing_age_df = housing_age_df.rename(columns={HOUSING_YEAR_COL: "median_year_built"})

print("Sentinel value counts before cleaning:")
print(housing_age_df["median_year_built"].value_counts().reindex(ACS_SENTINEL_VALUES, fill_value=0))

# ACS suppresses unreliable medians with sentinel codes rather than leaving them blank
# these must be treated as missing data, not as real years, and cleared before any calculations

housing_age_df["median_year_built"] = housing_age_df["median_year_built"].replace(ACS_SENTINEL_VALUES, np.nan)

print(f"\n{housing_age_df['median_year_built'].isna().sum():,} tracts now marked missing (were sentinel-coded)")

In [ ]:
# Step 5: load base tracts

tracts = gpd.read_file(BASE_TRACTS_PATH, engine="pyogrio")
tracts["GEOID"] = tracts["GEOID"].astype(GEOID_DTYPE).str.zfill(11)
if tracts.crs.to_string() != TARGET_CRS:
    tracts = tracts.to_crs(TARGET_CRS)

tract_area_m2 = tracts.set_index("GEOID").geometry.area
print(f"Base tracts loaded: {len(tracts):,} tracts across {tracts['city'].nunique()} cities")

In [ ]:
# Step 6: Set up helper functions

def compute_line_length_per_tract(line_path, tracts_gdf, target_crs=TARGET_CRS):
    lines = gpd.read_file(line_path, engine="pyogrio")
    if lines.crs.to_string() != target_crs:
        lines = lines.to_crs(target_crs)
    clipped = gpd.overlay(lines[["geometry"]], tracts_gdf[["GEOID", "geometry"]], how="intersection")
    clipped["length_m"] = clipped.geometry.length
    length_by_tract = clipped.groupby("GEOID")["length_m"].sum()
    del lines, clipped
    gc.collect()
    return length_by_tract

In [ ]:
# Step 7A: Calculate raw SNI values
        # Major roads length 
        # Measured in meters, negligible distortion from ESPG: 5070 projection which optimizes area

major_roads_length = compute_line_length_per_tract(MAJOR_ROADS_PATH, tracts)
tqdm.write(f"Major roads processed: {len(major_roads_length):,} tracts with road length > 0")

In [ ]:
# Step 7B: Calculate raw SNI values
        # Minor roads length 
        # Measured in meters, negligible distortion from ESPG: 5070 projection which optimizes area

minor_roads_length = compute_line_length_per_tract(MINOR_ROADS_PATH, tracts)
tqdm.write(f"Minor roads processed: {len(minor_roads_length):,} tracts with road length > 0")

In [ ]:
# Step 7C.1: Calculate raw SNI values
        # Railways length 
        # Measured in meters, negligible distortion from ESPG: 5070 projection which optimizes area

railway_length = compute_line_length_per_tract(RAILWAYS_PATH, tracts)
tqdm.write(f"Railways processed: {len(railway_length):,} tracts with rail length > 0")

In [ ]:
# Step 7C.2: Calculate raw SNI values
        # Public transit rail lines length 
        # Measured in meters, negligible distortion from ESPG: 5070 projection which optimizes area


transit_lines_length = compute_line_length_per_tract(TRANSIT_LINES_PATH, tracts)
tqdm.write(f"Transit lines processed: {len(transit_lines_length):,} tracts with rail length > 0")

In [ ]:
# Step 7D: Calculate raw SNI values
        # Overture POIs

pois = gpd.read_file(OVERTURE_POI_PATH, engine="pyogrio")
if pois.crs.to_string() != TARGET_CRS:
    pois = pois.to_crs(TARGET_CRS)

poi_joined = gpd.sjoin(pois[["geometry"]], tracts[["GEOID", "geometry"]], predicate="within")
poi_count = poi_joined.groupby("GEOID").size()

del pois, poi_joined
gc.collect()
tqdm.write(f"Overture POIs processed: {len(poi_count):,} tracts with at least one POI")

In [ ]:
# Step 7E: Calculate raw SNI values for zoning land coverage
        # One GeoPackage per city: loaded, combined, and overlaid against tracts
        # Computes total area (m2) and percent of tract area for every category
            # (commercial, industrial, institutional, mixed-use, open_space, other, residential, and blank/unclassified)
        # Only industiral, commercial, and mixed-use use downstream

zoning_files = glob.glob(os.path.join(ZONING_DIR, "*.gpkg"))
if not zoning_files:
    raise FileNotFoundError(f"No .gpkg files found in {ZONING_DIR}")

zoning_parts = []
for path in tqdm(zoning_files, desc="Loading zoning files"):
    part = gpd.read_file(path, engine="pyogrio")
    if part.crs.to_string() != TARGET_CRS:
        part = part.to_crs(TARGET_CRS)
    zoning_parts.append(part[[ZONING_CATEGORY_COLUMN, "geometry"]])

zoning = pd.concat(zoning_parts, ignore_index=True)
zoning = gpd.GeoDataFrame(zoning, geometry="geometry", crs=TARGET_CRS)
del zoning_parts
gc.collect()

# Normalize category labels: lowercase, spaces/hyphens -> underscores, blanks -> "unclassified"
zoning[ZONING_CATEGORY_COLUMN] = (
    zoning[ZONING_CATEGORY_COLUMN].astype(str).str.strip().str.lower().str.replace(r"[\s\-]+", "_", regex=True)
)
zoning[ZONING_CATEGORY_COLUMN] = zoning[ZONING_CATEGORY_COLUMN].replace(
    {"": "unclassified", "nan": "unclassified", "none": "unclassified"}
)

# Check and repair invalid geometries before overlay
n_invalid_zoning = (~zoning.geometry.is_valid).sum()
n_invalid_tracts = (~tracts.geometry.is_valid).sum()
tqdm.write(f"Invalid geometries -- zoning: {n_invalid_zoning:,} / tracts: {n_invalid_tracts:,}")

zoning["geometry"] = zoning.geometry.make_valid()
tracts["geometry"] = tracts.geometry.make_valid()

zoning_overlay = gpd.overlay(zoning, tracts[["GEOID", "geometry"]], how="intersection")
zoning_overlay["area_m2"] = zoning_overlay.geometry.area

zoning_area_by_category = (
    zoning_overlay.groupby(["GEOID", ZONING_CATEGORY_COLUMN])["area_m2"].sum().unstack(fill_value=0)
)
zoning_pct_by_category = zoning_area_by_category.div(tract_area_m2, axis=0) * 100

del zoning, zoning_overlay
gc.collect()

tqdm.write(
    f"Zoning processed: {len(zoning_area_by_category):,} tracts, "
    f"categories found: {zoning_area_by_category.columns.tolist()}"
)

In [ ]:
# Step 7F: Calculate raw SNI values
        # F: Join prepped CSV files

impervious_df = pd.read_csv(IMPERVIOUS_CSV_PATH, dtype={"tract_fips20": str})
impervious_df = impervious_df.rename(columns={"tract_fips20": "GEOID"})
impervious_df["GEOID"] = impervious_df["GEOID"].str.zfill(11)

lodes_df = pd.read_csv(LODES_CSV_PATH, dtype={"tract_geoid": str})
lodes_df = lodes_df.rename(columns={"tract_geoid": "GEOID"})
lodes_df["GEOID"] = lodes_df["GEOID"].str.zfill(11)

housing_age_df = pd.read_csv(HOUSING_AGE_CSV_PATH, dtype={"GEOID": str})
housing_age_df["GEOID"] = housing_age_df["GEOID"].str.zfill(11)
housing_age_df = housing_age_df.rename(columns={HOUSING_YEAR_COL: "median_year_built"})

# Confirm sentinel code have been removed before running
housing_age_df["median_year_built"] = housing_age_df["median_year_built"].replace(ACS_SENTINEL_VALUES, np.nan)

housing_age_df["building_age_years"] = CURRENT_YEAR - housing_age_df["median_year_built"]

tqdm.write(f"Impervious surface rows: {len(impervious_df):,}")
tqdm.write(f"LODES rows: {len(lodes_df):,}")
tqdm.write(f"Median year rows: {len(housing_age_df):,}")
tqdm.write(f"Housing age rows: {len(housing_age_df):,}")

In [ ]:
# Step 8: Merge all values onto base tracts

# confirm other data isn't dropped
master = tracts[["STATEFP", "COUNTYFP", "GEOID", "city", "ALAND", "AWATER", "point_count", "geometry"]].copy()

# Spatially-derived variables
master = master.merge(major_roads_length.rename("major_roads_length"), on="GEOID", how="left")
master = master.merge(minor_roads_length.rename("minor_roads_length"), on="GEOID", how="left")
master = master.merge(railway_length.rename("railway_length"), on="GEOID", how="left")
master = master.merge(transit_lines_length.rename("transit_lines_length"), on="GEOID", how="left")
master = master.merge(poi_count.rename("poi_count"), on="GEOID", how="left")

# Zoning: merge in both area (and percent-of-tract for every category
zoning_area_renamed = zoning_area_by_category.add_prefix("zoning_area_m2_").reset_index()
zoning_pct_renamed = zoning_pct_by_category.add_prefix("pct_").reset_index()
master = master.merge(zoning_area_renamed, on="GEOID", how="left")
master = master.merge(zoning_pct_renamed, on="GEOID", how="left")

zoning_cols = list(zoning_area_renamed.columns.drop("GEOID")) + list(zoning_pct_renamed.columns.drop("GEOID"))

# Tracts with no roads, railways, or POIs should be filled with 0, not NaN
spatial_zero_fill_cols = [
    "major_roads_length", "minor_roads_length", "railway_length", "poi_count",
] + zoning_cols
master[spatial_zero_fill_cols] = master[spatial_zero_fill_cols].fillna(0)

spatial_zero_fill_cols = [
    "major_roads_length", "minor_roads_length", "railway_length", "poi_count",
    ]
master[spatial_zero_fill_cols] = master[spatial_zero_fill_cols].fillna(0)

# External tabular variables: flagged, NOT auto-filled, since a gap could be real data issue
master = master.merge(impervious_df[["GEOID", IMPERVIOUS_VALUE_COL]], on="GEOID", how="left")
master = master.merge(lodes_df[["GEOID", LODES_VALUE_COL]], on="GEOID", how="left")
master = master.merge(housing_age_df[["GEOID", "median_year_built"]], on="GEOID", how="left")
master = master.merge(housing_age_df[["GEOID", "building_age_years"]], on="GEOID", how="left")

for col in [IMPERVIOUS_VALUE_COL, LODES_VALUE_COL, "median_year_built", "building_age_years"]:
    master[f"flag_{col}_missing"] = master[col].isna()

print(f"Master table assembled: {len(master):,} tracts, {master['city'].nunique()} cities")
print("----------")
print("Count of Null value tracts per column")
print(master.isna().sum())

In [ ]:
# Sanity check: confirm data compilation

print(f"Rows: {master.shape[0]}")
print(f"Columns: {master.shape[1]}")
print("--------------")
print(f"Categories: {master.columns.tolist()}")
print("--------------")
master.sample(5)

In [ ]:
# Step 9: Export master geopackage and CSV

master.to_file(MASTER_GPKG_OUTPUT_PATH, driver="GPKG", engine="pyogrio")
master.drop(columns="geometry").to_csv(MASTER_CSV_OUTPUT_PATH, index=False)

tqdm.write(f"Master GeoPackage exported to {MASTER_GPKG_OUTPUT_PATH}")
tqdm.write(f"Master CSV exported to {MASTER_CSV_OUTPUT_PATH}")

In [ ]:
# Step 10: Compile and export city summary table

zoning_area_cols = [c for c in master.columns if c.startswith("zoning_area_m2_")]
zoning_pct_cols = [c for c in master.columns if c.startswith("pct_")]

summary_rows = []

for city, city_df in tqdm(master.groupby("city"), desc="Summarizing by city"):
    row = {
        "city": city,
        "n_tracts": len(city_df),
        "major_roads_length_mean": city_df["major_roads_length"].mean(),
        "minor_roads_length_mean": city_df["minor_roads_length"].mean(),
        "railway_length_mean": city_df["railway_length"].mean(),
        "transit_lines_length_mean": city_df["transit_lines_length"].mean(),        
        "poi_count_mean": city_df["poi_count"].mean(),
        f"{IMPERVIOUS_VALUE_COL}_mean": city_df[IMPERVIOUS_VALUE_COL].mean(),
        f"{LODES_VALUE_COL}_mean": city_df[LODES_VALUE_COL].mean(),
        "median_year_built_mean": city_df["median_year_built"].mean(),
        "building_age_years_mean": city_df["building_age_years"].mean(),
        "major_roads_length_median": city_df["major_roads_length"].median(),
        "minor_roads_length_median": city_df["minor_roads_length"].median(),
        "railway_length_median": city_df["railway_length"].median(),
        "transit_lines_length_mean": city_df["transit_lines_length"].median(),        
        "poi_count_median": city_df["poi_count"].median(),
        f"{IMPERVIOUS_VALUE_COL}_median": city_df[IMPERVIOUS_VALUE_COL].median(),
        f"{LODES_VALUE_COL}_median": city_df[LODES_VALUE_COL].median(),
        "median_year_built_median": city_df["median_year_built"].median(),
        "building_age_years_median": city_df["building_age_years"].median(),
        f"n_missing_{IMPERVIOUS_VALUE_COL}": city_df[f"flag_{IMPERVIOUS_VALUE_COL}_missing"].sum(),
        f"n_missing_{LODES_VALUE_COL}": city_df[f"flag_{LODES_VALUE_COL}_missing"].sum(),
        "n_missing_building_age": city_df["flag_building_age_years_missing"].sum(),
    }

    for col in zoning_pct_cols:
        row[f"{col}_mean"] = city_df[col].mean()
        row[f"{col}_median"] = city_df[col].median()

    for col in zoning_area_cols:
        row[f"{col}_mean"] = city_df[col].mean()
        row[f"{col}_median"] = city_df[col].median()

    summary_rows.append(row)

prep_summary_df = pd.DataFrame(summary_rows).sort_values("city")
prep_summary_df.to_csv(PREP_SUMMARY_OUTPUT_PATH, index=False)

print(f"Data prep summary exported to {PREP_SUMMARY_OUTPUT_PATH}")
prep_summary_df